# Day 3: building an ANN from scratch (for real this time)

so days 1 and 2 were basically warmups, tiny networks on made up data just to see the gears turning. this one is the real deal, using the actual news headlines dataset from week 2 and trying to classify them into 6 categories, same as the classical ML stuff (decision tree, random forest, etc). except this time it's a neural net i built from scratch with numpy, no sklearn model doing the heavy lifting.

new stuff going into this one: batch normalization, dropout, and early stopping. all still hand-coded, no pytorch/tensorflow yet (that's tomorrow).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("data/news_dataset_cleaned.csv")

X_train_text, X_test_text, y_train_labels, y_test_labels = train_test_split(
    df["Title"], df["Category"], test_size=0.2, random_state=42
)

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (88, 649)
Test shape: (23, 649)


same bag-of-words setup as every week 2 classification day, just using sklearn's CountVectorizer for that part since turning text into numbers isn't really the point today, the network is.

In [2]:
##prepare the labels for training and testing
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(df["Category"]) ##fits the full datasets catergories, not just the training split
y_train_encoded = label_encoder.transform(y_train_labels)
y_test_encoded = label_encoder.transform(y_test_labels)

num_classes = len(label_encoder.classes_)

def one_hot(labels, num_classes): ##builds the all-zero grid and then sets the appropriate index to 1 for each label
    encoded = np.zeros((len(labels), num_classes))
    encoded [np.arange(len(labels)), labels] = 1
    return encoded

y_train = one_hot(y_train_encoded, num_classes)
y_test = one_hot(y_test_encoded, num_classes)
print("y_train shape:", y_train.shape)
print("Classes:", label_encoder.classes_)

y_train shape: (88, 6)
Classes: ['Business' 'Energy' 'Health' 'Markets' 'Politics' 'Technology']


labels are text ("Business", "Markets" etc) so they gotta get turned into numbers, and since we're using softmax at the end (which spits out a probability per class), the true labels need to look like that too -- basically a row of 6 numbers where only one of them is a 1 and the rest are 0. that's what one-hot means.

fit the label encoder on the WHOLE dataset's categories, not just train, because energy and health barely have any examples and sometimes don't even show up in one of the splits. this way we always get all 6 columns no matter what.

In [3]:
##intialize the network weights
np.random.seed(42)

##architecture: input (649 features) -> hidden1 (32, with batchnorm+relu+dropout) -> hidden2 (16, relu only) -> output (6 classes, softmax)
input_size = X_train.shape[1]
hidden1_size = 32
hidden2_size = 16
output_size = num_classes

W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2 / input_size) ##He initialization for relu networks to keep activation variance roughly consistent from layer to layer
b1 = np.zeros((1, hidden1_size))

gamma1 = np.ones(hidden1_size) ##gamma and beta are parameters for batch normalization, initialized to 1 and 0 respectively
beta1 = np.zeros(hidden1_size)

##second hidden layer weights, same He initialization pattern, no gamma/beta since this layer skips batchnorm
W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2 / hidden1_size)
b2 = np.zeros((1, hidden2_size))

##output layer weights, produces 1 raw score per class before softmax turns them into probabilities
W3 = np.random.randn(hidden2_size, output_size) * np.sqrt(2 / hidden2_size)
b3 = np.zeros((1, output_size))

dropout_rate = 0.2 ##dropout rate for regularization, randomly sets 20% of the neurons to zero during training to prevent overfitting

so this network has 3 layers total: 649 inputs (one per word in the vocab) -> 32 hidden neurons -> 16 hidden neurons -> 6 outputs (one per category).

one thing that's different from day 1/2: weights can't start at 0 here, they gotta be random, otherwise every neuron in a layer would literally learn the exact same thing forever (nothing to make them different from each other). the `np.sqrt(2 / input_size)` part isn't just a random guess like `* 0.5` was before, it's an actual real technique called He initialization, made specifically for relu networks so the numbers don't blow up or shrink to nothing as they pass through a bunch of layers.

gamma/beta are for batch norm (more on that in a sec), they start at 1 and 0 so batch norm basically does nothing extra at first until training adjusts them.

In [4]:
##forward pass functions
##need a version of softmax that can handle batch inputs, so we can compute the softmax for each sample in the batch over its own 6 class scores, independently. This is done by subtracting the max value in each row from each row before exponentiating, which prevents overflow and ensures numerical stability.
def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True)) ##axis=1 and keepdims=True makes every operation happen per row rather than the whole array
    return expZ / np.sum(expZ, axis=1, keepdims=True) ##^^^-np.max(Z, axis=1, keepdims=True) is numerical stability, so if a score of very large value is present, it doesn't blow up the exponentiation and cause NaNs

def relu(x): ##same ReLU from Day 2 -- this is a new file, so it has to be defined again here
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float) ##derivative of ReLU is 1 for positive inputs, 0 for negative inputs

day 2's softmax only worked on one row of numbers at a time. now we've got 88 samples going through at once, and each one needs its own softmax over its own 6 scores, so had to tweak it to work per-row instead of on the whole thing at once. the subtracting-the-max part is just to stop things from blowing up to infinity if a raw score happens to be some huge number, doesn't change the actual answer.

In [5]:
def forward(X, training=True):
    Z1 = X @ W1 + b1 ##first linear layer: raw (pre-activation, pre-batchnorm) output of hidden layer 1

    batch_mean = np.mean(Z1, axis=0) ##average down each column (feature) to get the mean for each feature across the batch
    batch_var = np.var(Z1, axis=0) ##same idea as batch_mean, but the variance for each feature across the batch
    Z1_norm = (Z1 - batch_mean) / np.sqrt(batch_var +1e-8) ##rescales each neuron's values to have 0 mean and unit variance
    Z1_bn = gamma1 * Z1_norm + beta1 ##re-applies the learnable scale/shift so the network isn't forced to keep strict 0 mean/1 variance if something else works better

    A1 = relu(Z1_bn) ##activation applied to the batch-normalized values, not the raw Z1

    if training:
        dropout_mask = (np.random.rand(*A1.shape) > dropout_rate).astype(float) ##80% are True, 20% are False, then converted to float (1.0 and 0.0) basically dropped
        A1_dropped = A1 * dropout_mask / (1 - dropout_rate) ##"inverted dropout": scales up the surviving 80% so the layer's overall output magnitude stays about the same whether dropout is on or off
    else:
        dropout_mask = np.ones_like(A1) ##at test time dropout is turned off completely, every neuron stays active
        A1_dropped = A1 ##no scaling needed here since training already compensated for it

    Z2 = A1_dropped @ W2 +b2 ##second hidden layer, linear step
    A2 = relu(Z2) ##second hidden layer, activation -- deliberately no batchnorm/dropout here for contrast against hidden layer 1

    Z3 = A2 @ W3 + b3 ##output layer: 1 raw score per class, before softmax
    A3 = softmax(Z3) ##converts the 6 raw scores per sample into a probability distribution that sums to 1

    cache = (Z1, batch_mean, batch_var, Z1_norm, Z1_bn, A1, dropout_mask, A1_dropped, Z2, A2, Z3, A3) ##bundling up every intermediate value computed above, since backprop needs each of these exact values to work backward through the network
    return A3, cache

ok this is the big one, let me try to break down what's actually happening layer by layer:

1. linear layer 1 (Z1) -- just the normal weighted sum + bias thing
2. batch norm -- take Z1 and basically "normalize" it using the mean/variance of the current batch, so the numbers stay in a nice consistent range instead of drifting all over the place. then gamma/beta let the network undo that a bit if it wants to
3. relu -- same as day 2, kills anything negative
4. dropout (training only) -- randomly turns off 20% of the neurons so the network can't lean too hard on any one of them. at test time this gets skipped entirely, we want the full network making the actual prediction
5. layer 2 -- just a normal relu layer, no batchnorm or dropout here, wanted to see a "plain" layer next to the fancy one
6. layer 3 (output) -- turns into a probability per category with softmax

we're also saving basically every intermediate value in `cache` because the backward pass (coming up) needs all of it to figure out the gradients.

In [6]:
##loss function- cross entropy loss, softmax's natural partner in crime
def compute_loss(y_true, y_pred):
    epsilon = 1e-8 ##small constant to prevent log(0) which is undefined
    return -np.mean(np.sum(y_true * np.log(y_pred + epsilon), axis=1)) ##for each sample, sum the log probabilities of the true class (y_true is one-hot encoded, so only the true class contributes to the sum), then average across all samples in the batch

predictions, cache = forward(X_train, training=False) ##forward pass on the training data, but with dropout turned off since we're not training yet
loss = compute_loss(y_train, predictions) ##compute the loss between the true labels and the predicted probabilities
print("Initial loss on training data:", loss)

Initial loss on training data: 2.7300205870362917


cross entropy instead of MSE this time since we're doing classification with softmax now, not predicting a number. basically it just checks "how confident was the model in the CORRECT answer" and punishes it hard if it was confidently wrong.

initial loss came out to like 2.73, kinda expected -log(1/6) which is around 1.79 (that's what you'd get if it was totally random/uniform guessing) but since the weights are random (not zero like day1/2) the untrained guesses aren't uniform, they can be randomly confident about wrong stuff, so a bit higher than 1.79 isn't weird at all.

In [7]:
##backward pass functions
def backward(X, y_true, cache):
    N = X.shape[0] ##number of samples in the batch
    Z1, batch_mean, batch_var, Z1_norm, Z1_bn, A1, dropout_mask, A1_dropped, Z2, A2, Z3, A3 = cache ##unpack the cached values from the forward pass

    dZ3 = (A3 - y_true) / N ##derivative of the loss w.r.t. the output layer's raw scores, averaged over the batch
    dW3 = A2.T @ dZ3 ##gradient of the loss w.r.t. the weights of the output layer
    db3 = np.sum(dZ3, axis=0, keepdims=True) ##gradient of the loss w.r.t. the biases of the

    ##hidden layer 2 backprop
    dA2 = dZ3 @ W3.T ##gradient of the loss w.r.t. the activations of hidden layer 2
    dZ2 = dA2 * relu_derivative(Z2) ##gradient of the loss w.r.t. the raw scores of hidden layer 2
    dW2 = A1_dropped.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True) ##gradient of the loss w.r.t. the biases of hidden layer 2

    ##backprop through dropout and relu into hidden layer 1
    dA1_dropped = dZ2 @ W2.T ##gradient of the loss
    dA1 = dA1_dropped * dropout_mask / (1 - dropout_rate) ##backprop through dropout, scaling the surviving neurons' gradients
    dZ1_bn = dA1 * relu_derivative(Z1_bn)

    dgamma1 = np.sum(dZ1_bn * Z1_norm, axis=0)
    dbeta1 = np.sum(dZ1_bn, axis=0)
    dZ1_norm = dZ1_bn * gamma1

    std_inv = 1 / np.sqrt(batch_var + 1e-8)
    dvar = np.sum(dZ1_norm * (Z1 - batch_mean) * -0.5 * std_inv**3, axis=0)
    dmean = np.sum(dZ1_norm * -std_inv, axis=0) + dvar * np.mean(-2 * (Z1 - batch_mean), axis=0)
    dZ1 = dZ1_norm * std_inv + dvar * 2 * (Z1 - batch_mean) / N + dmean / N

    ##last piece of the backward pass: gradients for the first layer's weights and biases
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return dW1, db1, dgamma1, dbeta1, dW2, db2, dW3, db3

honestly the hardest part of the whole project so far. going backward from the output:

- softmax + cross entropy together turn into just `predictions - y_true`, which is actually a really clean/nice result considering how complicated both of those look separately
- then it's the same pattern over and over: push the error back through a layer's weights, gate it through relu's derivative (kills gradient wherever a neuron was off), repeat
- dropout backward just reuses the same random mask from the forward pass, whatever got zeroed out then gets zero gradient now too, makes sense
- batch norm backward is the messy one. because the mean and variance are computed across the WHOLE batch, every single sample's output is kind of tangled up with every other sample's. so the gradient here has to account for "if i nudge this one value, it also nudges the shared mean and variance a little, which then nudges everyone else too." honestly didn't derive this fully from scratch on my own, it's a pretty well known formula, just went with the standard version and understood the general idea rather than proving every step by hand

In [8]:
##training loop- early stopping based on validation loss
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)
print("Sub-train shape:", X_train_sub.shape)
print("Validation shape:", X_val.shape)

Sub-train shape: (70, 649)
Validation shape: (18, 649)


early stopping needs its own held-out chunk of data to check "is this actually getting better" -- and it can't be the test set, that has to stay untouched till the very end. so this carves a slice out of the training data itself. only 88 training rows total so this leaves a pretty tiny validation set (18 rows), not a lot to go on but it's the right idea methodology-wise.

In [9]:
##Adam optimizer 
mW1, vW1 = np.zeros_like(W1), np.zeros_like(W1)
mb1, vb1 = np.zeros_like(b1), np.zeros_like(b1)
mgamma1, vgamma1 = np.zeros_like(gamma1), np.zeros_like(gamma1)
mbeta1, vbeta1 = np.zeros_like(beta1), np.zeros_like(beta1)
mW2, vW2 = np.zeros_like(W2), np.zeros_like(W2)
mb2, vb2 = np.zeros_like(b2), np.zeros_like(b2)
mW3, vW3 = np.zeros_like(W3), np.zeros_like(W3)
mb3, vb3 = np.zeros_like(b3), np.zeros_like(b3)

adam_beta1, adam_beta2, adam_epsilon = 0.9, 0.999, 1e-8

def adam_update(param, grad, m, v, t, learning_rate):
    m = adam_beta1 * m + (1 - adam_beta1) * grad
    v = adam_beta2 * v + (1 - adam_beta2) * (grad ** 2)
    m_hat = m / (1 - adam_beta1 ** t)
    v_hat = v / (1 - adam_beta2 ** t)
    param -= learning_rate * m_hat / (np.sqrt(v_hat) + adam_epsilon)
    return param, m, v

same adam from day 2, just way more parameter groups this time (8 instead of 4) since there's more layers plus the batch norm gamma/beta. named adam's own hyperparameters `adam_beta1`/`adam_beta2` instead of just `beta1`/`beta2` this time because `beta1` was already taken by batch norm's shift parameter, would've been a nasty silent bug otherwise.

In [10]:
##reset weights and Adam state to their original initialization before this second run, so it's a fair, separate comparison against the patience=20 run instead of continuing from already-trained weights
np.random.seed(42)
W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2 / input_size)
b1 = np.zeros((1, hidden1_size))
gamma1 = np.ones(hidden1_size)
beta1 = np.zeros(hidden1_size)
W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2 / hidden1_size)
b2 = np.zeros((1, hidden2_size))
W3 = np.random.randn(hidden2_size, output_size) * np.sqrt(2 / hidden2_size)
b3 = np.zeros((1, output_size))

mW1, vW1 = np.zeros_like(W1), np.zeros_like(W1)
mb1, vb1 = np.zeros_like(b1), np.zeros_like(b1)
mgamma1, vgamma1 = np.zeros_like(gamma1), np.zeros_like(gamma1)
mbeta1, vbeta1 = np.zeros_like(beta1), np.zeros_like(beta1)
mW2, vW2 = np.zeros_like(W2), np.zeros_like(W2)
mb2, vb2 = np.zeros_like(b2), np.zeros_like(b2)
mW3, vW3 = np.zeros_like(W3), np.zeros_like(W3)
mb3, vb3 = np.zeros_like(b3), np.zeros_like(b3)

quick note here, i actually ran this whole thing twice: once with `patience = 20` and once with `patience = 100` (patience = how many epochs it'll tolerate with no improvement before giving up). this cell resets everything back to the original random starting point so the second run is a fair comparison and not just continuing on top of the first run's already-trained weights. only the `patience = 100` version is what's actually in the code below, but i talk about both results down in the takeaway.

In [11]:
##full training loop
learning_rate = 0.01
epochs = 1000
patience = 100
best_val_loss = np.inf
patience_counter = 0
best_params = None

train_losses = []
val_losses = []

for epoch in range(1, epochs +1):
    predictions, cache = forward(X_train_sub, training=True)
    loss = compute_loss(y_train_sub, predictions)
    train_losses.append(loss)

    dW1, db1, dgamma1, dbeta1, dW2, db2, dW3, db3 = backward(X_train_sub, y_train_sub, cache)

    W1, mW1, vW1 = adam_update(W1, dW1, mW1, vW1, epoch, learning_rate)
    b1, mb1, vb1 = adam_update(b1, db1, mb1, vb1, epoch, learning_rate)
    gamma1, mgamma1, vgamma1 = adam_update(gamma1, dgamma1, mgamma1, vgamma1, epoch, learning_rate)
    beta1, mbeta1, vbeta1 = adam_update(beta1, dbeta1, mbeta1, vbeta1, epoch, learning_rate)
    W2, mW2, vW2 = adam_update(W2, dW2, mW2, vW2, epoch, learning_rate)
    b2, mb2, vb2 = adam_update(b2, db2, mb2, vb2, epoch, learning_rate)
    W3, mW3, vW3 = adam_update(W3, dW3, mW3, vW3, epoch, learning_rate)
    b3, mb3, vb3 = adam_update(b3, db3, mb3, vb3, epoch, learning_rate)

    val_predictions, _ = forward(X_val, training=False)
    val_loss = compute_loss(y_val, val_predictions)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_params = {
            "W1": W1.copy(), "b1": b1.copy(), "gamma1": gamma1.copy(), "beta1": beta1.copy(),
            "W2": W2.copy(), "b2": b2.copy(),
            "W3": W3.copy(), "b3": b3.copy()
        }
    else:
        patience_counter += 1

    if epoch % 50 == 0:
        print(f"Epoch {epoch}: train_loss={loss:.4f}, val_loss={val_loss:.4f}")

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
        break
W1, b1, gamma1, beta1 = best_params["W1"], best_params["b1"], best_params["gamma1"], best_params["beta1"]
W2, b2, W3, b3 = best_params["W2"], best_params["b2"], best_params["W3"], best_params["b3"]

print("\nBest validation loss:", f"{best_val_loss:.4f}")

Epoch 50: train_loss=0.0139, val_loss=1.3681
Epoch 100: train_loss=0.0062, val_loss=1.5085

Early stopping at epoch 111 (no improvement for 100 epochs)

Best validation loss: 0.9719


ok so a couple bugs happened while building this that are worth mentioning honestly:

1. first attempt, the early stopping check accidentally got attached to the "print every 50 epochs" check instead of its own real check. so it was just unconditionally stopping at epoch 50 every time no matter what patience was set to, which is why patience=20 and patience=100 gave the EXACT same result the first time i tried this. fixed it by separating the print-progress check from the actual `patience_counter >= patience` check.

2. once that was actually fixed, you can see what's really going on: train loss basically goes to 0 (like 0.006, fully memorized the 70 training rows) while val loss gets WORSE the longer it trains (best was 0.9719 super early on, around epoch 11, then it just climbs from there). classic overfitting, and giving it way more patience (100 instead of 20) didn't help at all, same best val loss both times, since the model peaks that early no matter what.

In [12]:
#Compare ANN against week 2 classical ML results 
test_predictions, _ = forward(X_test, training=False) ##turns off dropout so fully trained network makes the precition
test_loss = compute_loss(y_test, test_predictions)
test_accuracy = np.mean(np.argmax(test_predictions, axis=1) == np.argmax(y_test, axis=1)) #picks out which class has the highest probability for each sample, then compares it to the true class and averages over all samples to get accuracy

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

Test loss: 1.7255043930855514
Test accuracy: 0.391304347826087


## takeaway

test accuracy came out to about 39%. not great, and honestly worse than every single classical model from week 2 -- decision tree and gradient boosting both got 70%, even the "bad" ones like logistic regression and SVM got 61%. so the fancy from-scratch neural net with batch norm + dropout + adam + early stopping lost to a plain decision tree with zero tuning.

but i don't think this means the code is wrong (the math checks out, loss really did go down during training), i think it's more that this network is just too big for how little data we have. it's got around 21,500 learnable numbers in it (mostly from that first 649x32 layer) and only 70 actual training rows after splitting off validation. that's an insane amount of "brain" for barely any data to learn from. trees can just go "if the headline has the word stock in it, guess markets" super directly and cheaply, but a neural net has to sort of discover useful patterns on its own through a ton of tiny gradient nudges, which needs way more examples to actually work well.

also tried giving it more patience (letting it train longer before giving up) just to make sure it wasn't stopping too early, and it made zero difference to the final result, actually made it overfit harder (train loss near 0, val loss climbing). so this isn't a "just train it longer" problem, it's a "not enough data for a model this big" problem.

kind of ties back to stuff from week 2 too -- XGBoost lost to plain gradient boosting, randomizedsearchcv picked a worse model than gridsearchcv despite similar cv scores. same vibe here again: more advanced/complicated tool =/= automatically better, especially when the dataset is this small.